# Notebook 06: Geographic Bottleneck Analysis

## Objective
This notebook identifies geographic areas where shipping performance is inefficient.

### Key Tasks:
- Analyze performance by region and state
- Identify high lead-time areas
- Detect high-volume + poor-performance zones
- Highlight congestion-prone regions

This step helps uncover operational bottlenecks in the logistics network.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
# Load featured dataset
ncd = pd.read_csv("../data/featured_nassau_data.csv")

# Ensure datetime
ncd["Order Date"] = pd.to_datetime(ncd["Order Date"])
ncd["Ship Date"] = pd.to_datetime(ncd["Ship Date"])

ncd.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time_Days,Shipping Lead Time,Factory,Route_State,Route_Region,Order Month,Order Year,Order Month Name,Delay Status,Shipping Speed Category
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2024-01-09,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,6,6,Wicked Choccy's,Wicked Choccy's → Texas,Wicked Choccy's → Interior,1,2024,January,Delayed,Slow
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,7,7,Wicked Choccy's,Wicked Choccy's → Illinois,Wicked Choccy's → Interior,1,2024,January,Delayed,Slow
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2024-01-09,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,5,5,Lot's O' Nuts,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior,1,2024,January,On-Time,Slow
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,7,7,Lot's O' Nuts,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior,1,2024,January,Delayed,Slow
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2024-01-12,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,7,7,Wicked Choccy's,Wicked Choccy's → Pennsylvania,Wicked Choccy's → Atlantic,1,2024,January,Delayed,Slow


## Region-Level Performance Analysis

In [3]:
region_summary = ncd.groupby("Region").agg(
    Total_Shipments=("Order ID", "count"),
    Avg_Lead_Time=("Shipping Lead Time", "mean"),
    Lead_Time_Std=("Shipping Lead Time", "std")
).reset_index()

region_summary["Avg_Lead_Time"] = region_summary["Avg_Lead_Time"].round(2)
region_summary["Lead_Time_Std"] = region_summary["Lead_Time_Std"].round(2)

region_summary

,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Std
0,Atlantic,2986,3.96,1.88
1,Gulf,1620,4.11,1.84
2,Interior,2335,4.08,1.86
3,Pacific,3253,3.95,1.87


## State-Level Performance Analysis

In [4]:
state_summary = ncd.groupby("State/Province").agg(
    Total_Shipments=("Order ID", "count"),
    Avg_Lead_Time=("Shipping Lead Time", "mean"),
    Lead_Time_Std=("Shipping Lead Time", "std")
).reset_index()

state_summary["Avg_Lead_Time"] = state_summary["Avg_Lead_Time"].round(2)
state_summary["Lead_Time_Std"] = state_summary["Lead_Time_Std"].round(2)

state_summary

,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Std
0,Alabama,61,4.18,1.88
1,Alberta,26,3.77,1.61
2,Arizona,224,3.90,1.96
3,Arkansas,60,4.23,1.69
4,British Columbia,22,4.77,1.51
5,California,2001,3.95,1.88
6,Colorado,182,3.74,1.87
7,Connecticut,82,3.49,2.10
8,Delaware,96,4.03,1.74
9,District Of Columbia,10,4.20,1.55


In [5]:
# Top slow regions
slow_regions = region_summary.sort_values(
    by="Avg_Lead_Time", ascending=False
).head(5)

slow_regions

,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Std
1,Gulf,1620,4.11,1.84
2,Interior,2335,4.08,1.86
0,Atlantic,2986,3.96,1.88
3,Pacific,3253,3.95,1.87


In [6]:
# Top slow states
slow_states = state_summary.sort_values(
    by="Avg_Lead_Time", ascending=False
).head(10)

slow_states

,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Std
35,Newfoundland And Labrador,6,5.67,1.63
44,Prince Edward Island,10,5.40,1.43
53,Vermont,11,5.00,1.10
30,New Brunswick,8,5.00,1.20
38,Nova Scotia,6,4.83,1.94
4,British Columbia,22,4.77,1.51
12,Idaho,21,4.71,2.05
19,Maine,8,4.62,1.60
27,Montana,15,4.60,2.06
47,Saskatchewan,2,4.50,2.12


## High Volume + Poor Performance Analysis

In [7]:
# Define thresholds
volume_threshold = state_summary["Total_Shipments"].quantile(0.75)
lead_time_threshold = state_summary["Avg_Lead_Time"].quantile(0.75)

# Identify bottleneck states
bottleneck_states = state_summary[
    (state_summary["Total_Shipments"] >= volume_threshold) &
    (state_summary["Avg_Lead_Time"] >= lead_time_threshold)
]

bottleneck_states

,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Std
23,Michigan,255,4.26,1.94
50,Tennessee,183,4.28,1.65


In [8]:
#Flag Bottlenecks

state_summary["Bottleneck_Flag"] = np.where(
    (state_summary["Total_Shipments"] >= volume_threshold) &
    (state_summary["Avg_Lead_Time"] >= lead_time_threshold),
    "Bottleneck",
    "Normal"
)

state_summary.head()

,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Bottleneck_Flag
0,Alabama,61,4.18,1.88,Normal
1,Alberta,26,3.77,1.61,Normal
2,Arizona,224,3.90,1.96,Normal
3,Arkansas,60,4.23,1.69,Normal
4,British Columbia,22,4.77,1.51,Normal


In [9]:
# Region bottlenecks
region_volume_threshold = region_summary["Total_Shipments"].quantile(0.75)
region_lead_threshold = region_summary["Avg_Lead_Time"].quantile(0.75)

region_summary["Bottleneck_Flag"] = np.where(
    (region_summary["Total_Shipments"] >= region_volume_threshold) &
    (region_summary["Avg_Lead_Time"] >= region_lead_threshold),
    "Bottleneck",
    "Normal"
)

region_summary

,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Bottleneck_Flag
0,Atlantic,2986,3.96,1.88,Normal
1,Gulf,1620,4.11,1.84,Normal
2,Interior,2335,4.08,1.86,Normal
3,Pacific,3253,3.95,1.87,Normal


In [10]:
state_summary.sort_values("Avg_Lead_Time", ascending=False).head()

,State/Province,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Bottleneck_Flag
35,Newfoundland And Labrador,6,5.67,1.63,Normal
44,Prince Edward Island,10,5.40,1.43,Normal
53,Vermont,11,5.00,1.10,Normal
30,New Brunswick,8,5.00,1.20,Normal
38,Nova Scotia,6,4.83,1.94,Normal


In [11]:
region_summary.sort_values("Avg_Lead_Time", ascending=False)

,Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Bottleneck_Flag
1,Gulf,1620,4.11,1.84,Normal
2,Interior,2335,4.08,1.86,Normal
0,Atlantic,2986,3.96,1.88,Normal
3,Pacific,3253,3.95,1.87,Normal


In [12]:
state_summary.to_csv("../data/state_bottleneck_analysis.csv", index=False)
region_summary.to_csv("../data/region_bottleneck_analysis.csv", index=False)
bottleneck_states.to_csv("../data/high_risk_states.csv", index=False)

print("Geographic bottleneck datasets saved successfully.")

Geographic bottleneck datasets saved successfully.
